In [1]:
# Scenario: AI System for Detecting Lung Diseases from Chest X-rays
# 🚨 The Problem

# A large hospital receives thousands of chest X-rays daily.

# Radiologists are:

# Overworked

# Limited in number

# Required to make fast decisions

# Sometimes critical conditions like:

# 👉 Pneumonia
# 👉 COVID-19
# 👉 Normal lungs

# must be identified within minutes.

# Delays can cost lives.

# 💡 The Solution: Build an AI Assistant

# The hospital decides to deploy an AI model that can pre-screen X-rays and
# alert doctors.

# But there is a challenge…

# ❗ Medical datasets are usually SMALL.

# Training a deep neural network from scratch would require:

# Millions of labeled X-rays

# Massive GPU clusters

# Months of training

# Not practical.

# ⭐ Enter Transfer Learning (Your Code)

# Instead of starting from zero, engineers use:

# 👉 ResNet50 trained on ImageNet

# Although ImageNet contains everyday objects (dogs, cars, etc.),
# the early CNN layers learn universal visual patterns, like:

# ✅ Edges
# ✅ Gradients
# ✅ Textures
# ✅ Shapes

# These features are also present in medical scans.

# Import PyTorch library (main deep learning framework)
import torch

# Import torchvision models (contains pre-trained CNN architectures)
import torchvision.models as models

# Import neural network module from PyTorch
from torch import nn


# -------------------------------------------------------
# Step 1: Load a Pre-trained ResNet50 Model
# -------------------------------------------------------

# ResNet50 is a deep convolutional neural network with 50 layers.
# It is already trained on the ImageNet dataset (1.2 million images, 1000 classes).
# Using a pre-trained model allows us to reuse learned visual features
# such as edges, textures, shapes, and patterns.

model = models.resnet50(pretrained=True)


# -------------------------------------------------------
# Step 2: Freeze All Pre-trained Layers
# -------------------------------------------------------

# Transfer learning strategy:
# We freeze earlier layers so their weights do not change during training.
# These layers already learned general image features.

for param in model.parameters():
    param.requires_grad = False

# requires_grad = False means:
# - Gradients will NOT be computed
# - Weights will NOT be updated during backpropagation
# - Training becomes faster and needs less data


# -------------------------------------------------------
# Step 3: Modify the Final Classification Layer
# -------------------------------------------------------

# The original ResNet50 final layer predicts 1000 classes (ImageNet).
# Our problem only has 3 classes:
# 1. Normal
# 2. Pneumonia
# 3. COVID-19

num_classes = 3

# Replace the last fully connected layer (fc)
# model.fc.in_features gives the number of input features to the layer
# (2048 for ResNet50)

model.fc = nn.Linear(model.fc.in_features, num_classes)

# Now the model output will be 3 neurons instead of 1000.


# -------------------------------------------------------
# Step 4: Train Only the New Layer
# -------------------------------------------------------

# Since earlier layers are frozen,
# only the new fully connected layer will be trained.

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Trainable params: {trainable_params}")


# -------------------------------------------------------
# Expected Output Explanation
# -------------------------------------------------------

# The output will show the number of parameters that are trainable.
# Because we froze the backbone network, only the new classification
# layer parameters will be updated during training.

# This is a typical Transfer Learning workflow used in:
# - Medical Image Analysis
# - Face Recognition
# - Object Detection
# - Small dataset problems

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Trainable params: 6147


In [2]:
# Scenario: AI System for Detecting Fraud in Financial Transactions
# 🚨 The Problem
# A major bank processes millions of transactions daily.
# Fraud analysts are:
# - Overwhelmed by the sheer volume
# - Limited in number
# - Required to make instant decisions
# - Missing subtle fraud patterns hidden in massive datasets
# Critical cases like:
# 👉 Credit card fraud
# 👉 Money laundering
# 👉 Normal transactions
# must be flagged in real-time.
# Delays can cost millions and damage customer trust.

# 💡 The Solution: Build an AI Assistant
# The bank deploys an AI model that can pre-screen transactions and alert analysts.
# But here’s the challenge…
# ❗ Fraud datasets are usually imbalanced and small.
# Training a deep neural network from scratch would require:
# - Billions of labeled transactions
# - Huge compute clusters
# - Months of training
# Not practical.

# ⭐ Enter Transfer Learning (Your Code)
# Instead of starting from zero, engineers use:
# 👉 BERT (Bidirectional Encoder Representations from Transformers) trained on massive text corpora.
# Although BERT was trained on general language (Wikipedia, books, etc.), its early layers learn universal text patterns, like:
# - ✅ Word embeddings
# - ✅ Sentence structures
# - ✅ Semantic relationships
# - ✅ Contextual meaning
# These features are also present in transaction descriptions, merchant details, and customer behavior logs.
# By fine-tuning BERT on the bank’s fraud dataset, the AI can quickly learn to distinguish:
# - Suspicious vs. normal transactions
# - Fraud rings vs. genuine customer activity

# ⚡ Just like ResNet50 helps radiologists with X-rays, BERT helps fraud analysts with transaction logs —
# both leveraging transfer learning to solve problems where data is scarce but decisions are critical.




In [3]:
pip install transformers datasets torch scikit-learn

In [4]:
pip install --upgrade transformers

In [9]:
# !pip install transformers datasets torch scikit-learn

import torch
import numpy as np
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 🔹 1. Dataset (improved)
data = {
    "text": [
        "Payment to Amazon",
        "Transfer to offshore account",
        "Grocery store purchase",
        "Rapid withdrawals detected",
        "Salary credited",
        "Foreign suspicious transaction",
        "Multiple failed login attempts",
        "Large transfer at midnight",
        "Online shopping payment",
        "ATM withdrawal normal"
    ],
    "label": [0,1,0,1,0,1,1,1,0,0]
}

dataset = Dataset.from_dict(data)

# 🔹 2. Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

dataset = dataset.map(tokenize)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# 🔹 3. Split
train_test = dataset.train_test_split(test_size=0.3)
train_dataset = train_test["train"]
test_dataset = train_test["test"]

# 🔹 4. Model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# 🔹 5. Training Args (fixed for transformers 5.x)
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=5,
    eval_strategy="no",   # ❌ disable buggy evaluation
    save_strategy="no"
)

# 🔹 6. Trainer (no tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

# 🔹 7. Train
trainer.train()

# 🔹 8. ✅ Manual Evaluation (NO BUG)
predictions = trainer.predict(test_dataset)

logits = predictions.predictions
labels = predictions.label_ids

preds = np.argmax(logits, axis=1)

print("\nManual Evaluation:")
print("Accuracy:", accuracy_score(labels, preds))
print("Precision:", precision_score(labels, preds, zero_division=0))
print("Recall:", recall_score(labels, preds, zero_division=0))
print("F1 Score:", f1_score(labels, preds, zero_division=0))

# 🔹 9. Prediction function
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()
    return "Fraud" if pred == 1 else "Normal"

# 🔹 10. Test Predictions
print("\nPredictions:")
print("1.", predict("Unusual large transfer to foreign account"))
print("2.", predict("Paid electricity bill"))

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packag

Step,Training Loss



Manual Evaluation:
Accuracy: 0.3333333333333333
Precision: 0.0
Recall: 0.0
F1 Score: 0.0

Predictions:
1. Fraud
2. Normal


In [ ]:
#  The Solution: Build an AI Assistant
# The university deploys an AI model that can pre-screen essays and alert professors about suspicious content.
# But here’s the challenge…
# ❗ Academic plagiarism datasets are usually small and domain-specific.
# Training a deep NLP model from scratch would require:
# - Millions of labeled essays
# - Huge compute resources
# - Months of training
# Not practical.

# ⭐ Enter Transfer Learning (Your Code)
# Instead of starting from zero, engineers use:
# 👉 GPT-style language models trained on massive text corpora (books, articles, Wikipedia).
# Although these corpora contain general language, the early layers learn universal text features, like:
# - ✅ Grammar patterns
# - ✅ Semantic meaning
# - ✅ Sentence structures
# - ✅ Contextual relationships
# These features are also present in student essays.
# By fine-tuning the model on a smaller plagiarism dataset, the AI can quickly learn to distinguish:
# - Original vs. copied content
# - Paraphrased vs. genuinely written text
# - Citation misuse vs. proper referencing

# ⚡ Just like ResNet50 helps radiologists with X-rays, and BERT helps fraud analysts with transactions, GPT-style models help
# educators with essays — all leveraging transfer learning to solve problems where data is scarce but accuracy is critical.


In [11]:
# !pip install transformers torch scikit-learn

import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

# 🔹 Load GPT-style embedding model
model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# 🔹 Function to get sentence embedding
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)

    # Mean pooling
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.numpy()

# 🔹 Function to compare two essays
def check_plagiarism(text1, text2):
    emb1 = get_embedding(text1)
    emb2 = get_embedding(text2)

    similarity = cosine_similarity(emb1, emb2)[0][0]

    print(f"\nSimilarity Score: {similarity:.2f}")

    if similarity > 0.75:
        print("⚠️ Suspicious (Possible Plagiarism)")
    else:
        print("✅ Original Content")

# 🔹 Example Essays
essay1 = "Climate change is affecting the entire world due to global warming."
essay2 = "Global warming is impacting the world and causing climate change."

essay3 = "I wrote this essay based on my own understanding of the topic."

# 🔹 Test Cases
check_plagiarism(essay1, essay2)  # similar
check_plagiarism(essay1, essay3)  # different

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Similarity Score: 0.90
⚠️ Suspicious (Possible Plagiarism)

Similarity Score: 0.14
✅ Original Content


In [14]:
# !pip install sentence-transformers scikit-learn

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 🔹 Load pretrained model (GPT-style embedding)
model = SentenceTransformer('all-MiniLM-L6-v2')

# 🔹 Function to check plagiarism
def check_plagiarism(text1, text2):
    emb1 = model.encode([text1])
    emb2 = model.encode([text2])

    similarity = cosine_similarity(emb1, emb2)[0][0]

    print(f"\nSimilarity Score: {similarity:.2f}")

    if similarity > 0.7:
        print("⚠️ Suspicious (Plagiarism)")
    else:
        print("✅ Original Content")

# 🔹 Test Examples
essay1 = "Climate change is affecting the entire world due to global warming."
essay2 = "Global warming is impacting the world and causing climate change."
essay3 = "I wrote this essay based on my own ideas."

check_plagiarism(essay1, essay2)  # similar
check_plagiarism(essay1, essay3)  # different

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Similarity Score: 0.90
⚠️ Suspicious (Plagiarism)

Similarity Score: 0.09
✅ Original Content


In [12]:
# Install required libraries if not already installed:
# pip install transformers datasets peft accelerate

# Install required libraries if not already installed:
# pip install transformers datasets peft accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset
from peft import get_peft_model, LoraConfig

# 1. Load base model and tokenizer
model_name = "gpt2"   # you can replace with another causal LM
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure pad token exists (GPT-2 doesn't have one by default)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Define LoRA configuration
lora_config = LoraConfig(
    r=8,                  # rank of low-rank matrices
    lora_alpha=32,        # scaling factor
    lora_dropout=0.1,     # dropout for regularization
    bias="none",          # no bias terms updated
    task_type="CAUSAL_LM",# language modeling task
    target_modules=["c_attn"]  # GPT-2 attention projection layers
)

# Wrap model with LoRA adapters
model = get_peft_model(model, lora_config)

# 3. Create a tiny synthetic dataset
train_texts = ["Hello world", "LoRA fine-tuning is fun", "Transformers are powerful"]
eval_texts = ["Testing evaluation", "Another sample"]

train_dataset = Dataset.from_dict({"text": train_texts})
eval_dataset = Dataset.from_dict({"text": eval_texts})

# Tokenization function with labels
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )
    tokens["labels"] = tokens["input_ids"].copy()  # labels required for loss
    return tokens

# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 4. Training configuration
training_args = TrainingArguments(
    output_dir="./lora-finetuned-model",   # save directory
    per_device_train_batch_size=2,         # small batch size
    gradient_accumulation_steps=2,         # accumulate gradients
    learning_rate=2e-4,                    # tuned for LoRA
    num_train_epochs=1,                    # demo run
    logging_steps=1,                       # log every step
    save_strategy="epoch",                 # save at end of epoch
    fp16=True                              # mixed precision training
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

# 6. Train
trainer.train()

# 7. Save only LoRA adapter weights (few MB instead of GBs!)
model.save_pretrained("./lora-weights")

print("Training complete. LoRA weights saved in ./lora-weights")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,8.129440


Training complete. LoRA weights saved in ./lora-weights


In [13]:
# Big Idea Behind This Code

# Instead of shipping a 500MB–10GB model every time…

# You ship:

# 👉 A tiny adapter (few MB)

# Then dynamically attach it to the base model.

# This enables:

# ✅ Modular AI systems
# ✅ Fast deployment
# ✅ Cheap storage
# ✅ Multi-domain adapters

# This pattern is exploding across the industry.

# 🚀 Real-World Scenario (VERY Powerful)
# 🛒 Scenario: E-Commerce Company Builds Multiple AI Assistants

# An e-commerce giant wants different AI behaviors:

# Sentiment analyzer

# Product recommender

# Customer support bot

# Return-policy assistant

# But retraining separate LLMs would cost millions.

# So engineers do something smart:

# 👉 One Base Model
# 👉 Multiple LoRA Adapters


# Install required libraries if not already installed:
# pip install transformers datasets peft accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset
from peft import get_peft_model, LoraConfig, PeftModel

# 1. Load base model and tokenizer
model_name = "gpt2"   # you can replace with another causal LM
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure pad token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Define LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["c_attn"]  # GPT-2 uses 'c_attn' for QKV projection
)

# Wrap model with LoRA
model = get_peft_model(model, lora_config)

# 3. Create a tiny synthetic dataset
train_texts = ["Hello world", "LoRA fine-tuning is fun", "Transformers are powerful"]
eval_texts = ["Testing evaluation", "Another sample"]

train_dataset = Dataset.from_dict({"text": train_texts})
eval_dataset = Dataset.from_dict({"text": eval_texts})

# Tokenization function with labels
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )
    tokens["labels"] = tokens["input_ids"].copy()  # labels required for loss
    return tokens

train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 4. Training configuration
training_args = TrainingArguments(
    output_dir="./lora-finetuned-model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=1,   # keep small for demo
    logging_steps=1,
    save_strategy="epoch",
    fp16=True
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

# 6. Train
trainer.train()

# 7. Save only LoRA adapter weights (few MB instead of GBs!)
model.save_pretrained("lora-sentiment")  # folder will contain adapter_config.json

print("Training complete. LoRA weights saved in ./lora-sentiment")

# 8. Reload base model and attach adapter for inference
base_model = AutoModelForCausalLM.from_pretrained("gpt2")
sentiment_model = PeftModel.from_pretrained(base_model, "lora-sentiment")

# Example inference
input_text = "I love using transformers and LoRA adapters!"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids
output_ids = sentiment_model.generate(input_ids, max_length=50)
print("Output:", tokenizer.decode(output_ids[0], skip_special_tokens=True))


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,8.129440


Training complete. LoRA weights saved in ./lora-sentiment


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Output: I love using transformers and LoRA adapters!

I love using transformers and LoRA adapters! I love using transformers and LoRA adapters! I love using transformers and LoRA adapters! I love using transformers and LoRA
